# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR\u02c6<sup>2</sup> dataset using the `mlcroissant` library, including field and record references by `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id` references.

The FAIR\u02c6<sup>2</sup> dataset contains multiple record sets and columns. We will enumerate them below:

In [ ]:
# List available record sets by @id and name
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} | Name: {rs.get('name', '<no name>')}")
    print("  Fields (columns):")
    for fld in rs.get('fields', []):
        print(f"    - @id: {fld['@id']} | name: {fld.get('name','<no name>')} | dataType: {fld.get('dataType','<no type>')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. Use the record set and field `@id` values from the overview.

In [ ]:
# Choose which record sets to extract
# We'll use all available record sets by their @id for demonstration
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Loaded record set: {record_set_id} | shape: {df.shape}')

# Example: print columns of the first record set
if record_set_ids:
    print(f"\nColumns for record set '{record_set_ids[0]}':")
    print(dataframes[record_set_ids[0]].columns.tolist())
    print('\nSample records:')
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We will work with the main protein quantification matrix, filtering and normalizing one of its quantitative columns. *Replace the `numeric_field_id` and `group_field_id` below with concrete `@id`s for the fields you want to analyze, referencing the overview above.*

In [ ]:
# --- Replace these with the actual @id values as found above ---
# Example, suppose '@id': 'protein_quant_matrix' for record set,
#             'abundance' for numeric field, 'sample_group' for group field (these must match the dataset)

# For demonstration, we take the first record set and its first numeric field
record_set_id = record_set_ids[0] if record_set_ids else None

df = dataframes[record_set_id] if record_set_id else None

# List all fields for selection
if record_set_id:
    print(f"Available DataFrame columns for {record_set_id}:")
    print(df.columns.tolist())

    # Try to auto-detect a numeric field (float/int)
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
    else:
        # Try to coerce a likely column
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                numeric_field = col
                break
            except Exception:
                continue
    # Select a group field if any categorical exist
    group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique()<30]
    group_field = group_field_candidates[0] if group_field_candidates else None

    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Group by group_field if available
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print("No categorical group field found for grouping.")
else:
    print('No record sets found in dataset.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll do a histogram and a boxplot for the selected numeric field. You may adjust the field as appropriate for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if record_set_id and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
We've loaded the FAIR\u02c6<sup>2</sup> dataset, explored its schema and contents by `@id`, performed filtering, normalization, grouping, and visualized distributions for quantitative fields.

You can extend this notebook by adding more detailed analysis, deeper cleaning, or new visualizations tailored to your use case. Refer to the dataset's documentation for additional context on record set and field meanings.